# Data Preprocessing for Fuzzy Movie Recommendation System

This notebook demonstrates the data preprocessing pipeline for the MovieLens dataset, including:
- Loading and exploring the dataset
- Data cleaning and transformation
- Feature engineering
- Creating user-item matrices
- Computing movie popularity and genre encodings

In [ ]:
# Import required libraries
import sys
import os
sys.path.append('../backend')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Import our custom modules
from data.load_movielens import MovieLensLoader
from data.preprocessing import MovieLensPreprocessor
from utils import setup_logging

# Setup
setup_logging()
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")

## 1. Load MovieLens Dataset

We'll start by loading the MovieLens dataset. For demonstration purposes, we'll use a sample dataset, but you can switch to the full 10M dataset by changing the parameters.

In [ ]:
# Initialize the MovieLens loader
loader = MovieLensLoader(data_dir='../backend/data')

# Option 1: Create sample dataset for quick testing
print("Creating sample dataset...")
sample_path = '../backend/data/sample_data.pkl'
loader.create_sample_dataset(sample_path, n_users=1000, n_movies=500)

# Load the sample data
import pickle
with open(sample_path, 'rb') as f:
    raw_data = pickle.load(f)

movies_df = raw_data['movies']
ratings_df = raw_data['ratings']

print(f"Loaded {len(movies_df)} movies and {len(ratings_df)} ratings")
print(f"Users: {ratings_df['user_id'].nunique()}")
print(f"Rating range: {ratings_df['rating'].min()} - {ratings_df['rating'].max()}")

In [ ]:
# Uncomment below to use real MovieLens dataset instead
# Note: This will download and process the full dataset

# print("Downloading MovieLens 1M dataset...")
# dataset_path = loader.download_dataset('1M')
# movies_df = loader.load_movies(dataset_path, '1M')
# ratings_df = loader.load_ratings(dataset_path, '1M', sample_size=100000)  # Sample for demo
# users_df = loader.load_users(dataset_path, '1M')

# print(f"Loaded {len(movies_df)} movies and {len(ratings_df)} ratings")

## 2. Exploratory Data Analysis

In [ ]:
# Display basic information about the datasets
print("=== MOVIES DATASET ===")
print(movies_df.head())
print(f"\nShape: {movies_df.shape}")
print(f"Columns: {movies_df.columns.tolist()}")

print("\n=== RATINGS DATASET ===")
print(ratings_df.head())
print(f"\nShape: {ratings_df.shape}")
print(f"Columns: {ratings_df.columns.tolist()}")

In [ ]:
# Analyze genre distribution
all_genres = []
for genres in movies_df['genres']:
    if isinstance(genres, list):
        all_genres.extend(genres)
    elif isinstance(genres, str):
        all_genres.extend(genres.split('|'))

genre_counts = pd.Series(all_genres).value_counts()

plt.figure(figsize=(12, 6))
genre_counts.head(15).plot(kind='bar')
plt.title('Top 15 Movie Genres Distribution')
plt.xlabel('Genre')
plt.ylabel('Number of Movies')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Total unique genres: {len(genre_counts)}")
print(f"Top 10 genres:\n{genre_counts.head(10)}")

In [ ]:
# Analyze rating distribution
plt.figure(figsize=(15, 5))

# Rating distribution
plt.subplot(1, 3, 1)
ratings_df['rating'].hist(bins=20, edgecolor='black')
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Frequency')

# Ratings per user
plt.subplot(1, 3, 2)
user_rating_counts = ratings_df['user_id'].value_counts()
user_rating_counts.hist(bins=30, edgecolor='black')
plt.title('Ratings per User Distribution')
plt.xlabel('Number of Ratings')
plt.ylabel('Number of Users')
plt.yscale('log')

# Ratings per movie
plt.subplot(1, 3, 3)
movie_rating_counts = ratings_df['movie_id'].value_counts()
movie_rating_counts.hist(bins=30, edgecolor='black')
plt.title('Ratings per Movie Distribution')
plt.xlabel('Number of Ratings')
plt.ylabel('Number of Movies')
plt.yscale('log')

plt.tight_layout()
plt.show()

print(f"Average rating: {ratings_df['rating'].mean():.2f}")
print(f"Rating std: {ratings_df['rating'].std():.2f}")
print(f"Average ratings per user: {user_rating_counts.mean():.1f}")
print(f"Average ratings per movie: {movie_rating_counts.mean():.1f}")

## 3. Data Preprocessing Pipeline

In [ ]:
# Initialize the preprocessor
preprocessor = MovieLensPreprocessor()

print("Starting preprocessing pipeline...")

# Run the complete preprocessing pipeline
preprocessed_data = preprocessor.preprocess_full_dataset(
    movies_df, ratings_df, users_df=None
)

print("Preprocessing completed!")

## 4. Analyze Preprocessed Data

In [ ]:
# Examine movie popularity scores
movies_processed = preprocessed_data['movies']

plt.figure(figsize=(15, 5))

# Popularity distribution
plt.subplot(1, 3, 1)
movies_processed['popularity'].hist(bins=30, edgecolor='black')
plt.title('Movie Popularity Distribution')
plt.xlabel('Popularity Score (0-100)')
plt.ylabel('Number of Movies')

# Year distribution
plt.subplot(1, 3, 2)
movies_processed['year'].dropna().hist(bins=20, edgecolor='black')
plt.title('Movie Year Distribution')
plt.xlabel('Year')
plt.ylabel('Number of Movies')

# Popularity vs Year
plt.subplot(1, 3, 3)
plt.scatter(movies_processed['year'], movies_processed['popularity'], alpha=0.6)
plt.title('Popularity vs Release Year')
plt.xlabel('Year')
plt.ylabel('Popularity Score')

plt.tight_layout()
plt.show()

print(f"Popularity stats:")
print(movies_processed['popularity'].describe())

In [ ]:
# Examine top popular movies
top_popular = movies_processed.nlargest(10, 'popularity')[['title', 'genres', 'popularity', 'year']]
print("Top 10 Most Popular Movies:")
print(top_popular.to_string(index=False))

# Examine least popular movies
least_popular = movies_processed.nsmallest(10, 'popularity')[['title', 'genres', 'popularity', 'year']]
print("\nTop 10 Least Popular Movies:")
print(least_popular.to_string(index=False))

In [ ]:
# Analyze user-item matrix
user_item_matrix = preprocessed_data['user_item_matrix']
print(f"User-item matrix shape: {user_item_matrix.shape}")
print(f"Matrix density: {user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.4f}")
print(f"Total ratings: {user_item_matrix.nnz}")

# Visualize matrix sparsity (sample)
sample_matrix = user_item_matrix[:50, :50].toarray()

plt.figure(figsize=(10, 8))
plt.imshow(sample_matrix, cmap='Blues', aspect='auto')
plt.title('User-Item Matrix Sparsity (50x50 sample)')
plt.xlabel('Movies')
plt.ylabel('Users')
plt.colorbar(label='Rating')
plt.show()

## 5. Genre Encoding Analysis

In [ ]:
# Analyze genre encoding
genre_encoder = preprocessed_data['genre_encoder']
genre_matrix = preprocessed_data['genre_matrix']

print(f"Encoded genres: {genre_encoder.classes_}")
print(f"Genre matrix shape: {genre_matrix.shape}")

# Show genre co-occurrence
genre_cooccurrence = np.dot(genre_matrix.T, genre_matrix)
genre_cooccurrence_df = pd.DataFrame(
    genre_cooccurrence, 
    index=genre_encoder.classes_, 
    columns=genre_encoder.classes_
)

plt.figure(figsize=(12, 10))
sns.heatmap(genre_cooccurrence_df, annot=True, fmt='d', cmap='Blues')
plt.title('Genre Co-occurrence Matrix')
plt.tight_layout()
plt.show()

## 6. Test Genre Match Computation

In [ ]:
# Test genre match computation with sample user preferences
sample_preferences = {
    'Action': 'Very High',
    'Comedy': 'Medium',
    'Romance': 'Low',
    'Thriller': 'High',
    'Sci-Fi': 'Very High',
    'Drama': 'Medium',
    'Horror': 'Very Low'
}

print("Sample user preferences:")
for genre, pref in sample_preferences.items():
    print(f"  {genre}: {pref}")

# Compute genre match scores for all movies
genre_match_scores = preprocessor.compute_genre_match_score(
    sample_preferences, genre_matrix
)

# Add scores to movies dataframe for analysis
movies_with_scores = movies_processed.copy()
movies_with_scores['genre_match'] = genre_match_scores

# Show top matches
top_matches = movies_with_scores.nlargest(10, 'genre_match')[['title', 'genres', 'genre_match', 'popularity']]
print("\nTop 10 Genre Matches:")
print(top_matches.to_string(index=False))

# Visualize genre match distribution
plt.figure(figsize=(10, 6))
plt.hist(genre_match_scores, bins=30, edgecolor='black', alpha=0.7)
plt.title('Genre Match Score Distribution')
plt.xlabel('Genre Match Score (0-1)')
plt.ylabel('Number of Movies')
plt.axvline(genre_match_scores.mean(), color='red', linestyle='--', label=f'Mean: {genre_match_scores.mean():.3f}')
plt.legend()
plt.show()

## 7. Save Preprocessed Data

In [ ]:
# Save the preprocessed data
output_path = '../backend/data/preprocessed.pkl'
preprocessor.save_preprocessed(preprocessed_data, output_path)

print(f"Preprocessed data saved to: {output_path}")
print(f"File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

# Verify we can load it back
loaded_data = preprocessor.load_preprocessed(output_path)
print(f"Successfully loaded back {len(loaded_data['movies'])} movies")

## 8. Summary Statistics

In [ ]:
# Generate comprehensive summary
print("=== PREPROCESSING SUMMARY ===")
print(f"Total movies: {len(movies_processed)}")
print(f"Total ratings: {len(preprocessed_data['ratings'])}")
print(f"Total users: {len(preprocessed_data['user_to_idx'])}")
print(f"Unique genres: {len(genre_encoder.classes_)}")
print(f"User-item matrix density: {user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.4f}")
print(f"Average movie popularity: {movies_processed['popularity'].mean():.2f}")
print(f"Average rating: {preprocessed_data['ratings']['rating'].mean():.2f}")

print("\n=== GENRE DISTRIBUTION ===")
for i, genre in enumerate(genre_encoder.classes_):
    count = genre_matrix[:, i].sum()
    print(f"{genre}: {count} movies ({count/len(movies_processed)*100:.1f}%)")

print("\nPreprocessing completed successfully!")
print("The data is now ready for training fuzzy logic and ANN models.")